===========================================================================
# Hospital Performance Communication Application
===========================================================================

This notebook provides a lightweight demonstration of the final application. A user selects a
facility and may optionally provide a domain, measure, and communication style. Structured
retrieval returns the validated evidence, Python constructs the factual narrative, and the
language model changes only the communication style.


## Task 1: Imports and Setup
--------------------------------------------------------------------------


In [1]:
# Import libraries

import json
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [3]:
# Define file paths

narrative_dir = Path("Data/Narratives")
evaluation_dir = Path("Data/Evaluation")

# Load data

evidence_repository = pd.read_csv(narrative_dir / "evidence_repository.csv")
prompt_experiment = pd.read_csv(narrative_dir / "prompt_experiment.csv")
testing_results = pd.read_csv(evaluation_dir / "testing_results.csv")

with open(narrative_dir / "communication_prompts.json", "r", encoding="utf-8") as file:
    communication_prompts = json.load(file)

print("Evidence Rows:", len(evidence_repository))
print("Prompt Experiment Rows:", len(prompt_experiment))
print("Testing Results:", len(testing_results))
print("Communication Styles:", len(communication_prompts))

print("\nCommunication Styles:")
print(list(communication_prompts.keys()))

print("\nTesting Results by Audience:")
display(testing_results.groupby("Audience").size().rename("Narratives").reset_index())

Evidence Rows: 57298
Prompt Experiment Rows: 60
Testing Results: 20
Communication Styles: 4

Communication Styles:
['Patient Friendly', 'Executive Summary', 'Clinical', 'Community Report']

Testing Results by Audience:


,Audience,Narratives
0,Clinical,5
1,Community Report,5
2,Executive Summary,5
3,Patient Friendly,5


## Task 2: Application Functions
--------------------------------------------------------------------------


In [5]:
# Retrieve hospital evidence

def retrieve_hospital_evidence(evidence_repository, facility_name=None, facility_id=None,
                               measure=None, domain=None):

    results = evidence_repository.copy()

    if facility_name is not None:
        results = results[results["Facility Name"].str.contains(facility_name, case=False, na=False)]

    if facility_id is not None:
        results = results[results["Facility ID"].astype(str) == str(facility_id)]

    if measure is not None:
        results = results[results["Measure Name"].str.contains(measure, case=False, na=False)]

    if domain is not None:
        results = results[results["Domain"] == domain]

    return results.copy()

## Task 3: Generation Model
--------------------------------------------------------------------------


In [4]:
# Load the same generation model used during evaluation
generation_model_name = "google/flan-t5-large"

generation_tokenizer = AutoTokenizer.from_pretrained(generation_model_name)
generation_model = AutoModelForSeq2SeqLM.from_pretrained(generation_model_name)
generation_model.eval()

print("Generation Model:", generation_model_name)

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Generation Model: google/flan-t5-large


In [7]:
# Map out parameters

generation_parameters = {
    "max_new_tokens": 200,
    "min_new_tokens": 40,
    "do_sample": False,
    "num_beams": 2,
    "no_repeat_ngram_size": 2,
    "early_stopping": True
}

In [8]:
# Generate narrative text

def generate_text(prompt, max_new_tokens=200, min_new_tokens=40, do_sample=False,
                  num_beams=2, no_repeat_ngram_size=2, early_stopping=True):

    inputs = generation_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)

    with torch.no_grad():
        outputs = generation_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            min_new_tokens=min_new_tokens,
            do_sample=do_sample,
            num_beams=num_beams,
            no_repeat_ngram_size=no_repeat_ngram_size,
            early_stopping=early_stopping
        )

    return generation_tokenizer.decode(outputs[0], skip_special_tokens=True)

## Task 4: Final Application
--------------------------------------------------------------------------


In [12]:
# Retrieve prepared prompt from prompt experiment

def retrieve_prepared_prompt(facility_id, communication_style="Patient Friendly", prompt_version=1):

    prepared = prompt_experiment[
        (prompt_experiment["Facility ID"].astype(str) == str(facility_id)) &
        (prompt_experiment["Audience"] == communication_style) &
        (prompt_experiment["Prompt Version"] == prompt_version)
    ].copy()

    if prepared.empty:
        return None

    return prepared.iloc[0]

In [13]:
# Task 4: Generate hospital communication using prepared Notebook 4 prompts

def hospital_communication_app(facility_name=None, facility_id=None,
                               communication_style="Patient Friendly", prompt_version=1):

    if communication_style not in communication_prompts:
        raise ValueError(f"Communication style must be one of: {list(communication_prompts.keys())}")

    retrieved = retrieve_hospital_evidence(
        evidence_repository=evidence_repository,
        facility_name=facility_name,
        facility_id=facility_id
    )

    if retrieved.empty:
        return {
            "Evidence": retrieved,
            "Communication Style": communication_style,
            "Prompt Version": prompt_version,
            "Performance Evidence": None,
            "Context Evidence": None,
            "Prompt": None,
            "Generated Narrative": "No matching hospital evidence was found."
        }

    selected_facility_id = retrieved["Facility ID"].iloc[0]

    prepared = retrieve_prepared_prompt(
        facility_id=selected_facility_id,
        communication_style=communication_style,
        prompt_version=prompt_version
    )

    if prepared is None:
        return {
            "Evidence": retrieved,
            "Communication Style": communication_style,
            "Prompt Version": prompt_version,
            "Performance Evidence": None,
            "Context Evidence": None,
            "Prompt": None,
            "Generated Narrative": "No prepared prompt was found for this facility."
        }

    generated_narrative = generate_text(prepared["Prompt"], **generation_parameters)

    return {
        "Evidence": retrieved,
        "Communication Style": communication_style,
        "Prompt Version": prompt_version,
        "Performance Evidence": prepared["Performance Evidence"],
        "Context Evidence": prepared["Context Evidence"],
        "Prompt": prepared["Prompt"],
        "Generated Narrative": generated_narrative
    }

## Task 5: Demonstration
--------------------------------------------------------------------------


In [14]:
# Task 5: Demonstration

facility_name = prompt_experiment["Facility Name"].iloc[0]
communication_style = "Patient Friendly"
prompt_version = 1

result = hospital_communication_app(
    facility_name=facility_name,
    communication_style=communication_style,
    prompt_version=prompt_version
)

print("APPLICATION RESULT")
print("=" * 80)
print("Facility:", facility_name)
print("Communication Style:", result["Communication Style"])
print("Prompt Version:", result["Prompt Version"])

print("\nPERFORMANCE EVIDENCE")
print("-" * 80)
print(result["Performance Evidence"])

print("\nCONTEXT EVIDENCE")
print("-" * 80)
print(result["Context Evidence"])

print("\nGENERATED NARRATIVE")
print("-" * 80)
print(result["Generated Narrative"])

APPLICATION RESULT
Facility: MONUMENT HEALTH RAPID CITY HOSPITAL
Communication Style: Patient Friendly
Prompt Version: 1

PERFORMANCE EVIDENCE
--------------------------------------------------------------------------------
Facility: MONUMENT HEALTH RAPID CITY HOSPITAL

Strengths:
  Timely and Effective Care
  Relative Performance: 0.45 SD from cohort

  Key Measures:
    - ED Time - Psychiatric Patients: 179.00 Minutes
      Cohort Mean: 319.04 | Lower is Better
    - ED Time - All Patients: 162.00 Minutes
      Cohort Mean: 196.20 | Lower is Better

Weaknesses:
  Patient Survey

  Key Measures:
    - Recommend hospital: 3.00 Stars
      Cohort Mean: 3.41 | Higher is Better
    - Communication about medicines: 2.00 Stars
      Cohort Mean: 2.25 | Higher is Better

CONTEXT EVIDENCE
--------------------------------------------------------------------------------
nan

GENERATED NARRATIVE
--------------------------------------------------------------------------------
The MONUMENT HEALTH 

In [15]:
# Demonstrate all communication styles for one hospital

facility_name = prompt_experiment["Facility Name"].iloc[0]

for communication_style in communication_prompts.keys():

    result = hospital_communication_app(
        facility_name=facility_name,
        communication_style=communication_style,
        prompt_version=1
    )

    print("\n" + "=" * 100)
    print("COMMUNICATION STYLE:", communication_style)
    print("-" * 100)
    print(result["Generated Narrative"])


COMMUNICATION STYLE: Patient Friendly
----------------------------------------------------------------------------------------------------
The MONUMENT HEALTH RAPID CITY HOSPITAL is a hospital that provides timely and effective care. The hospital has low ED time and poor communication about medicines.

COMMUNICATION STYLE: Executive Summary
----------------------------------------------------------------------------------------------------
The facility at MONUMENT HEALTH RAPID CITY HOSPITAL has a strong focus on timely and effective care. The hospital has low patient satisfaction. However, the hospital's patient survey scores are below average.

COMMUNICATION STYLE: Clinical
----------------------------------------------------------------------------------------------------
Healthcare-Associated Infections - CLABSI (ICU + select wards): 6491.0 Device Days Healthcare Associated Diseases: 0.0 Cases, 0 Healthcare associated infections

COMMUNICATION STYLE: Community Report
--------------